# Enclave Inference Service — Gemma 3

A pre-packaged inference pipeline running **inside** the enclave, behind a tiny FastAPI endpoint.

- The **model owner** uploads Gemma 3 weights through syftbox; the pipeline loads them from the synced location.
- Every `/infer` request is logged into a dataset on the **enclave's own datasite** — the private log data never leaves the enclave (not even over Drive).
- A **researcher** who wants to analyse the logs submits a job that **both** data owners must approve.

In production this runs as the `enclave-model-api-example` docker image (see `packages/enclave-model-api-example/` — its `docker/` dir and `Justfile`). This notebook drives the same components in-process with an in-memory network.

## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@openmined.org` | Runs the pre-packaged inference pipeline + syftbox |
| **Model owner** | `model_owner@openmined.org` | Uploads the Gemma 3 weights dataset |
| **Log owner** | `log_owner@openmined.org` | Co-governs the inference logs (joint approval) |
| **Researcher** | `researcher@openmined.org` | Submits analysis jobs on the logs |

## Setup

Install the model deps (in docker these come from `packages/enclave-model-api-example/docker/requirements.txt`), authenticate to Kaggle, and download the Gemma 3 270m flax weights (cached after first run). Accept the Gemma license once at https://www.kaggle.com/models/google/gemma-3.

In [ ]:
!uv pip install gemma "kagglehub==1.0.0"

In [ ]:
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from fastapi.testclient import TestClient

from syft_enclaves import SyftEnclaveClient
from enclave_model_api.backend import GemmaBackend
from enclave_model_api.log_writer import LOG_FILE_NAME
from enclave_model_api.logs_dataset import ensure_logs_dataset
from enclave_model_api.paths import private_dataset_dir
from enclave_model_api.server import create_app
from enclave_model_api.service import InferenceService

MODEL_SIZE = "270m"
KAGGLE_HANDLE = "google/gemma-3/flax/gemma-3-270m-it"

In [ ]:
import kagglehub

kagglehub.login()
download_dir = Path(kagglehub.model_download(KAGGLE_HANDLE))
print(f"Weights directory: {download_dir}")
print(f"Contents: {os.listdir(download_dir)}")

---
## Prepare the model owner's weights dataset

The dataset layout the pre-packaged pipeline expects (see `enclave_model_api/backend.py`):

- `tokenizer.model` — SentencePiece tokenizer
- `<checkpoint>/` — the orbax checkpoint directory (exactly one subdirectory)

Note there is **no inference code** in the dataset — the pipeline ships inside the container. The mock side is just a model card.

In [ ]:
def create_model_private_dir() -> Path:
    private_dir = Path(tempfile.mkdtemp()) / f"gemma3-weights-{random.randint(1, 1_000_000)}"
    private_dir.mkdir(parents=True)
    shutil.copy(download_dir / "tokenizer.model", private_dir / "tokenizer.model")
    ckpt_dirs = [p for p in download_dir.iterdir() if p.is_dir()]
    shutil.copytree(ckpt_dirs[0], private_dir / ckpt_dirs[0].name)
    return private_dir


def create_model_mock_file() -> Path:
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True)
    p = tmp / "model_card.txt"
    p.write_text(f"Gemma 3 {MODEL_SIZE}-IT, served behind POST /infer on the enclave.")
    return p


model_private_dir = create_model_private_dir()
model_mock = create_model_mock_file()
print(f"Private weights dir: {model_private_dir}")

---
## Step 0 — Spin up the network

In [ ]:
enclave, model_owner, log_owner, researcher = SyftEnclaveClient.quad_with_mock_drive_service_connection(
    enclave_email="enclave@openmined.org",
    do1_email="model_owner@openmined.org",
    do2_email="log_owner@openmined.org",
    ds_email="researcher@openmined.org",
    use_in_memory_cache=False,
)
print(f"  Enclave     : {enclave.email}  (data owners: {enclave.data_owners})")

---
## Step 1 — Model owner uploads the weights through syftbox

In [ ]:
model_owner.create_dataset(
    name="gemma3_model",
    mock_path=model_mock,
    private_path=model_private_dir,
    summary=f"Gemma 3 {MODEL_SIZE.upper()}-IT weights",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("gemma3_model", enclave.email)
model_owner.sync()
enclave.sync()

---
## Step 2 — Enclave starts the inference service

In docker this happens automatically: the runner's `post_init` creates the logs dataset and `packages/enclave-model-api-example/docker/inference_server.py` builds the service from `SYFT_ENCLAVE_*` env vars. Here we do the same in-process.

The logs dataset lives on the **enclave's own datasite**: only the synthetic mock sample goes to Drive, the private `requests.jsonl` is the live log sink and is excluded from sync.

In [ ]:
logs_dir = ensure_logs_dataset(enclave, "inference_logs")

service = InferenceService(
    backend=GemmaBackend(),
    model_size=MODEL_SIZE,
    weights_dir=private_dataset_dir(enclave.syftbox_folder, model_owner.email, "gemma3_model"),
    logs_dir=logs_dir,
)
service.try_load()  # weights already synced; in docker a poll thread does this
http = TestClient(create_app(service))
http.get("/model-status").json()

---
## Step 3 — Run inference

Each request is answered by the real Gemma model and appended to the private log.

In [ ]:
for query in [
    "A nurse greeted the patient. Was the nurse most likely male or female? Why?",
    "What is the capital of the Netherlands?",
]:
    response = http.post("/infer", json={"query": query, "max_new_tokens": 64})
    print(f"Q: {query}\nA: {response.json()['completion']}\n")

In [ ]:
# The private log on the enclave — this file never leaves the enclave
print((logs_dir / LOG_FILE_NAME).read_text())

---
## Step 4 — Researcher discovers the logs dataset

The researcher only sees the mock (a synthetic schema sample), never the private records.

In [ ]:
enclave.sync()
researcher.sync()
researcher.datasets.get("inference_logs", datasite=enclave.email)

---
## Step 5 — Researcher submits an analysis job on the logs

The job runs **on the enclave**, where the private logs live. It only reports aggregates.

In [ ]:
job_code = f'''
import json
import os

import syft_client as sc

log_files = sc.resolve_dataset_files_path("inference_logs", owner_email="{enclave.email}")
log_file = [f for f in log_files if f.name == "{LOG_FILE_NAME}"][0]
records = [json.loads(line) for line in open(log_file).read().splitlines()]

os.makedirs("outputs", exist_ok=True)
with open("outputs/log_summary.json", "w") as f:
    json.dump({{
        "total_requests": len(records),
        "avg_elapsed": sum(r["stats"]["elapsed"] for r in records) / len(records),
    }}, f, indent=2)
'''

job_dir = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
job_dir.mkdir(parents=True)
(job_dir / "main.py").write_text(job_code)

researcher.submit_python_job(
    enclave.email,
    str(job_dir / "main.py"),
    "log_analysis",
    datasets={enclave.email: ["inference_logs"]},
)

---
## Step 6 — Both data owners must approve

The enclave distributes the job for review; it only becomes `approved` once **all** configured data owners have signed off.

In [ ]:
enclave.sync()
enclave.receive_jobs()

model_owner.sync()
log_owner.sync()
model_owner.approve_job(model_owner.jobs["log_analysis"])
enclave.sync()
print("after model_owner approval:", enclave.jobs["log_analysis"].status)

log_owner.approve_job(log_owner.jobs["log_analysis"])
enclave.sync()
print("after log_owner approval: ", enclave.jobs["log_analysis"].status)

---
## Step 7 — Run the job and fetch the results

Results go to the submitting researcher only — the data owners approve computations but never receive (or can download) the raw logs.

In [ ]:
enclave.run_jobs()
enclave.distribute_results()

researcher.sync()
researcher_job = researcher.jobs["log_analysis"]
print("status:", researcher_job.status)
print(json.load(open(researcher_job.output_paths[0])))

---
## Deploying for real

```bash
cd packages/enclave-model-api-example

# Local docker (dev token, no TEE):
just inference-local-build
just inference-local-run enclave@example.com do1@x.com,do2@y.com do1@x.com ../../credentials/token_enclave.json

# GCP Confidential Space:
just inference-build-push
just inference-start enclave@example.com do1@x.com
```

Then query it: `curl -X POST http://<ip>:8080/infer -H 'Content-Type: application/json' -d '{"query": "Hello"}'`

Note: the default `SYFT_ENCLAVE_FRESH_STATE=true` wipes the logs dataset on every boot — set it to `false` to persist logs across restarts.